# Día 1 — Del péndulo simple al caos determinista

### Taller: Física no lineal en el aula
**Congreso de Profesores de Física — Educación Secundaria**

---

## Cómo usar este cuaderno (leer antes de empezar)

Esto es un **cuaderno de Google Colab**. El código corre en una computadora de
Google, no en la suya: no hay que instalar nada.

**Los cuatro pasos del arranque:**

1. Arriba a la derecha, apretar **"Copiar en Drive"** (o *File → Save a copy in Drive*).
   👉 Si no hacen esto, pueden mirar pero **no guardar sus cambios**.
2. En el menú: **Entorno de ejecución → Ejecutar todas** (*Runtime → Run all*).
3. Si aparece un cartel que dice *"Este cuaderno no lo creó Google"*, apretar
   **"Ejecutar de todos modos"**.
4. Esperar ~20 segundos. Listo.

**Cómo se lee una celda:** cada bloque gris es una celda de código. A la izquierda
tiene un ▶. Si aparece `[ ]` no se ejecutó; si aparece `[3]` ya se ejecutó.

> ⚠️ **La regla de oro de Colab:** las celdas se ejecutan **en orden, de arriba
> hacia abajo**. Si algo da error raro, casi siempre es porque se salteó una celda.
> La solución universal: *Entorno de ejecución → Reiniciar y ejecutar todo*.

**Qué van a tener que hacer ustedes:**

- ✏️ **Para probar** → mover un deslizador y observar. No se escribe nada.
- 🧩 **Ejercicio** → cambiar **un número** en una celda y volver a ejecutarla (Shift+Enter).

No hace falta saber programar. Si algo no se entiende, se puede ignorar el código
y quedarse con los gráficos.

---
## 0. Preparación

Ejecutar esta celda una sola vez. No hay nada que instalar: Colab ya trae todo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk
from ipywidgets import interact, FloatSlider, IntSlider, SelectionSlider, Button, Output
from IPython.display import display

plt.rcParams["figure.figsize"] = (7, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

g = 9.81   # aceleración de la gravedad, m/s²
print("Todo listo ✓   numpy", np.__version__)

---
## 1. El péndulo con aproximación de ángulo pequeño

La ecuación exacta del péndulo es

$$\ddot{\theta} = -\frac{g}{L}\,\sin\theta$$

y es **no lineal** por culpa del $\sin\theta$. El truco de siempre: si $\theta$ es
chico, $\sin\theta \approx \theta$, y queda

$$\ddot{\theta} = -\omega_0^2\,\theta, \qquad \omega_0 = \sqrt{g/L}$$

que sabemos resolver a mano:

$$\theta(t) = \theta_0\cos(\omega_0 t), \qquad T_0 = 2\pi\sqrt{L/g}$$

Lo notable de esta solución: **el período no depende de la amplitud** (isocronía).
Eso es lo que hace que un reloj de péndulo funcione... y es *falso*, como vamos a
ver en dos minutos.

In [ ]:
@interact(L=FloatSlider(min=0.1, max=2.0, step=0.05, value=1.0,
                        description="L (m)", continuous_update=False),
          theta0_grados=IntSlider(min=1, max=170, value=15,
                        description="θ₀ (°)", continuous_update=False))
def pendulo_lineal(L, theta0_grados):
    th0 = np.radians(theta0_grados)
    w0  = np.sqrt(g/L)
    T0  = 2*np.pi/w0
    t   = np.linspace(0, 4*T0, 1000)
    plt.figure()
    plt.plot(t, th0*np.cos(w0*t), lw=2)
    plt.axhline(0, color="gray", lw=0.6)
    plt.xlabel("t (s)"); plt.ylabel("θ (rad)")
    plt.title(f"Solución lineal:  θ(t) = θ₀·cos(ω₀t)      T₀ = {T0:.3f} s")
    plt.show()

✏️ **Para probar:** mover el ángulo inicial de 1° a 170°. La forma de la curva no
cambia y el período tampoco. Guarden esa observación: la vamos a romper enseguida.

---
## 2. Resolver la ecuación exacta: un integrador en 8 líneas

Como no queremos aproximar, hay que integrar numéricamente. Usamos
**Runge–Kutta de orden 4 (RK4)**, que es lo que hay adentro de casi cualquier
simulador. La idea: en vez de avanzar con la pendiente del punto inicial (que
sería el método de Euler), se promedian cuatro estimaciones de la pendiente
dentro del paso.

No hace falta entenderlo en detalle para usarlo, pero está bueno que lo vean:
son ocho renglones y no hay magia adentro.

In [ ]:
def rk4(f, u0, tmax, dt):
    n  = int(tmax/dt)
    ts = np.linspace(0, n*dt, n+1)
    us = np.zeros((n+1, len(u0)))
    us[0] = u0
    for i in range(n):
        u, t = us[i], ts[i]
        k1 = f(u,             t)
        k2 = f(u + dt/2*k1,   t + dt/2)
        k3 = f(u + dt/2*k2,   t + dt/2)
        k4 = f(u + dt*k3,     t + dt)
        us[i+1] = u + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    return ts, us

El estado del péndulo es $u = (\theta, \omega)$. La ecuación de segundo orden se
escribe como dos de primer orden:

$$\dot{\theta} = \omega, \qquad
\dot{\omega} = -\frac{g}{L}\sin\theta \;-\; b\,\omega \;+\; A\cos(\Omega t)$$

Por ahora dejamos $b = 0$ (sin rozamiento) y $A = 0$ (sin forzado). Los vamos a
encender en la sección 5.

In [ ]:
def pendulo(u, t, L=1.0, b=0.0, A=0.0, Omega=2.0):
    theta, omega = u
    return np.array([omega,
                     -(g/L)*np.sin(theta) - b*omega + A*np.cos(Omega*t)])

---
## 3. ¿Cuándo se rompe la aproximación?

Superponemos la solución lineal (línea punteada) con la solución numérica exacta
(línea llena).

In [ ]:
@interact(theta0_grados=IntSlider(min=1, max=179, value=20,
                        description="θ₀ (°)", continuous_update=False),
          L=FloatSlider(min=0.1, max=2.0, step=0.05, value=1.0,
                        description="L (m)", continuous_update=False))
def comparacion(theta0_grados, L):
    th0 = np.radians(theta0_grados)
    w0  = np.sqrt(g/L)
    T0  = 2*np.pi/w0
    ts, us = rk4(lambda u, t: pendulo(u, t, L=L), [th0, 0.0], 5*T0, 0.002)
    plt.figure()
    plt.plot(ts, us[:, 0], lw=2.2, label="exacto (numérico)")
    plt.plot(ts, th0*np.cos(w0*ts), lw=2, ls="--", label="aproximación lineal")
    plt.axhline(0, color="gray", lw=0.6)
    plt.xlabel("t (s)"); plt.ylabel("θ (rad)")
    plt.title(f"θ₀ = {theta0_grados}°")
    plt.legend(loc="lower left")
    plt.show()

✏️ **Para probar:** empezar en 10° (se superponen, no se distinguen), pasar a 45°
(se empiezan a desfasar) y llegar a 170° (no tienen nada que ver).

Fíjense en algo importante: el error **no** es que la solución exacta sea "fea".
Sigue siendo periódica y perfectamente ordenada. Lo que falla es que
**el período depende de la amplitud**.

---
## 4. El período exacto: existe, pero no es elemental

Integrando la conservación de la energía se llega a

$$T(\theta_0) = 4\sqrt{\frac{L}{g}}\; K\!\left(\sin^2\frac{\theta_0}{2}\right)$$

donde $K$ es la **integral elíptica completa de primera especie**. O sea: *sí* hay
fórmula cerrada, pero involucra una función que no es elemental y que hay que
evaluar numéricamente igual.

Este es un buen momento para el mensaje de fondo del taller: la frontera relevante
no es *"tiene fórmula / no tiene fórmula"*, sino **qué preguntas puedo responder
con lo que tengo**.

Abajo comparamos tres cosas: la predicción lineal (que dice que $T$ es constante),
la fórmula elíptica, y el período **medido sobre la simulación** — detectando
cuándo el péndulo vuelve a pasar por donde arrancó.

In [ ]:
def T_exacto(theta0, L=1.0):
    "Período exacto vía integral elíptica. Ojo: scipy usa el parámetro m = k²."
    return 4*np.sqrt(L/g)*ellipk(np.sin(theta0/2)**2)

def periodo_medido(theta0, L=1.0, dt=1e-3):
    "Mide el período buscando dos pasajes consecutivos por ω = 0 subiendo."
    ts, us = rk4(lambda u, t: pendulo(u, t, L=L),
                 [theta0, 0.0], 6*2*np.pi*np.sqrt(L/g), dt)
    w = us[:, 1]
    cruces = []
    for i in range(1, len(ts)):
        if w[i-1] < 0 <= w[i]:                      # cambio de signo − → +
            a = -w[i-1]/(w[i] - w[i-1])             # interpolación lineal
            cruces.append(ts[i-1] + a*dt)
            if len(cruces) == 2:
                break
    return cruces[1] - cruces[0] if len(cruces) == 2 else np.nan

In [ ]:
T0 = 2*np.pi*np.sqrt(1.0/g)

th_curva = np.radians(np.arange(1, 176, 2))
th_pts   = np.radians(np.arange(10, 171, 20))

plt.figure()
plt.plot(np.degrees(th_curva), [T_exacto(t)/T0 for t in th_curva],
         lw=2.5, label="fórmula elíptica")
plt.scatter(np.degrees(th_pts), [periodo_medido(t)/T0 for t in th_pts],
            s=55, zorder=5, color="crimson", label="medido en la simulación")
plt.axhline(1, ls="--", color="gray", label="predicción lineal")
plt.xlabel("θ₀ (grados)"); plt.ylabel("T(θ₀) / T₀")
plt.title("El período sí depende de la amplitud")
plt.legend()
plt.show()

Los puntos rojos caen exactamente sobre la curva: la fórmula elíptica y la
simulación coinciden (en las pruebas, con error relativo del orden de $10^{-13}$).
La recta gris es lo que predice la aproximación lineal, y se despega cada vez más.

🧩 **Ejercicio 1.** ¿A partir de qué amplitud el error de la aproximación lineal
supera el **1 %**?

Cambien el número de la celda de abajo y vuelvan a ejecutarla (Shift+Enter) hasta
que el resultado dé 1.0 %.

In [ ]:
theta_prueba = 20      # ←←← CAMBIAR ESTE NÚMERO (en grados)

# ---- no hace falta tocar nada de acá para abajo ----
err = 100*(T_exacto(np.radians(theta_prueba))/T0 - 1)
print(f"Con θ₀ = {theta_prueba}°  el período real es un {err:.2f} % más largo")
print(f"que el que predice la fórmula lineal.")

> **Para el aula:** este número explica por qué los relojes de péndulo usan
> amplitudes chicas, y por qué en el laboratorio de secundaria conviene medir
> $T$ con $\theta_0 < 15°$ si se quiere verificar $T = 2\pi\sqrt{L/g}$.

---
## 5. El espacio de fases

En vez de graficar $\theta$ contra $t$, graficamos $\omega$ contra $\theta$. Cada
estado del sistema es **un punto**; su evolución es **una curva**. Todo el
comportamiento posible del sistema entra en un solo dibujo.

Las flechas grises son el **campo vectorial**: en cada punto indican hacia dónde
se mueve el sistema. Las trayectorias simplemente siguen las flechas.

Ahora encendemos las dos cosas que rompen la conservación de la energía:

- **$b$** — rozamiento (disipación),
- **$A$** — forzado externo $A\cos(\Omega t)$ (inyección de energía).

In [ ]:
@interact(b=FloatSlider(min=0.0, max=1.0, step=0.02, value=0.0,
                        description="b (roce)", continuous_update=False),
          A=FloatSlider(min=0.0, max=2.5, step=0.05, value=0.0,
                        description="A (forzado)", continuous_update=False),
          Omega=FloatSlider(min=0.2, max=4.0, step=0.05, value=2.0,
                        description="Ω", continuous_update=False),
          tmax=IntSlider(min=10, max=200, step=10, value=40,
                        description="t max (s)", continuous_update=False))
def espacio_de_fases(b, A, Omega, tmax):
    fig, ax = plt.subplots(figsize=(7.5, 5))

    # --- campo vectorial (dibujado sin el forzado, que depende de t) ---
    th = np.linspace(-3*np.pi, 3*np.pi, 25)
    om = np.linspace(-8, 8, 17)
    TH, OM = np.meshgrid(th, om)
    dTH = OM
    dOM = -g*np.sin(TH) - b*OM
    n = np.hypot(dTH, dOM) + 1e-9
    ax.quiver(TH, OM, dTH/n, dOM/n, color="gray", alpha=0.45, width=0.0022)

    # --- trayectorias desde varias condiciones iniciales ---
    for th0, om0 in [(-2.5,0), (0.5,0), (2.0,0), (0,4.0), (0,6.5), (0,-5.5)]:
        _, us = rk4(lambda u, t: pendulo(u, t, L=1.0, b=b, A=A, Omega=Omega),
                    [th0, om0], float(tmax), 0.005)
        ax.plot(us[:,0], us[:,1], lw=1.4)
        ax.plot(th0, om0, "ko", ms=4)

    ax.set_xlim(-3*np.pi, 3*np.pi); ax.set_ylim(-8, 8)
    ax.set_xlabel("θ (rad)"); ax.set_ylabel("ω (rad/s)")
    ax.set_title(f"Espacio de fases     b = {b:.2f}    A = {A:.2f}")
    plt.show()

✏️ **Para probar, en este orden:**

1. **$b = 0$, $A = 0$.** Curvas cerradas (oscilación) y curvas abiertas arriba y
   abajo (el péndulo da vueltas enteras). La curva que separa ambos regímenes es
   la **separatriz**, y pasa por el equilibrio inestable $\theta = \pi$.
2. **Subir $b$ a 0.3.** Las curvas se enroscan hacia $(0,0)$: el reposo es ahora
   un **atractor**. Todas las condiciones iniciales terminan ahí.
3. **$b = 0.3$ y subir $A$.** La energía que se pierde por rozamiento se repone
   desde afuera. Aparece un ciclo al que el sistema tiende.
4. **$b = 0.2$, $A \approx 1.5$, $\Omega \approx 2$, t max = 200.** La trayectoria
   deja de cerrarse sobre sí misma. Eso ya es caos, en el mismo péndulo de siempre.

> **El punto pedagógico:** *disipación ⟹ atractor*. Sin disipación no hay
> atractores, sólo órbitas que conservan la energía. Este es exactamente el
> concepto que necesitamos para la segunda parte.

---
## 6. El péndulo magnético: atractores y cuencas de atracción

Un péndulo largo con un imán en la punta, oscilando sobre **tres imanes fijos**
en el plano. Es un experimento de escritorio (se consigue armado como juguete) y
**no tiene solución analítica**.

Con el péndulo largo y oscilaciones no muy grandes, el modelo es un punto $(x,y)$
en el plano sometido a:

- una fuerza restitutiva hacia el centro, $-k\,\vec{r}$;
- rozamiento, $-b\,\dot{\vec{r}}$;
- la atracción de cada imán,
  $\sum_i \dfrac{\vec{r}_i - \vec{r}}{\left(|\vec{r}_i - \vec{r}|^2 + d^2\right)^{3/2}}$

donde $d$ es la altura a la que pasa el imán del péndulo sobre la mesa (y de paso
evita que la fuerza se vuelva infinita).

Hay **tres atractores**: los tres imanes. La pregunta interesante no es *"¿dónde
termina?"* sino **"¿en cuál de los tres, según dónde lo solté?"**

In [ ]:
# posiciones de los tres imanes (en los vértices de un triángulo equilátero)
IMANES  = np.array([[np.cos(a), np.sin(a)]
                    for a in (np.pi/2, np.pi/2 + 2*np.pi/3, np.pi/2 + 4*np.pi/3)])
COLORES = ["tomato", "steelblue", "mediumseagreen"]

# parámetros del modelo
K_RES, B_ROCE, D_ALT = 0.5, 0.05, 0.25

Acá viene un detalle técnico que vale la pena señalar, porque es la diferencia
entre que esto tarde 3 segundos o 3 minutos: la función de abajo integra
**todas las condiciones iniciales a la vez**, como vectores de numpy, en lugar de
hacer un `for` sobre cada punto. Es el mismo RK4 de antes, pero `S` es una matriz
de $4 \times N$ en vez de un vector de 4.

In [ ]:
def derivada(S, k=K_RES, b=B_ROCE, d=D_ALT):
    "S tiene forma (4, N): las filas son x, y, vx, vy de N péndulos a la vez."
    x, y, vx, vy = S
    ax = -k*x - b*vx
    ay = -k*y - b*vy
    for xm, ym in IMANES:
        r3 = ((x - xm)**2 + (y - ym)**2 + d*d)**1.5
        ax += (xm - x)/r3
        ay += (ym - y)/r3
    return np.stack([vx, vy, ax, ay])

def paso_rk4(S, dt, **kw):
    k1 = derivada(S,           **kw)
    k2 = derivada(S + dt/2*k1, **kw)
    k3 = derivada(S + dt/2*k2, **kw)
    k4 = derivada(S + dt*k3,   **kw)
    return S + dt/6*(k1 + 2*k2 + 2*k3 + k4)

def integrar(S, dt, tmax, guardar=False, cada=3, **kw):
    "Integrador simple. Si guardar=True devuelve tambien la trayectoria."
    traj = [S.copy()] if guardar else None
    for i in range(int(tmax/dt)):
        S = paso_rk4(S, dt, **kw)
        if guardar and i % cada == 0:
            traj.append(S.copy())
    return (S, np.array(traj)) if guardar else (S, None)

def integrar_rapido(S, dt, tmax, bloque=50, **kw):
    # Igual que integrar(), pero va sacando del calculo los pendulos que ya
    # se detuvieron sobre un iman. Da exactamente el mismo resultado y tarda
    # la mitad: es lo que hace usable el mapa de cuencas en clase.
    n = S.shape[1]
    final = np.empty((4, n))
    quedan = np.arange(n)
    for _ in range(0, int(tmax/dt), bloque):
        for _ in range(bloque):
            S = paso_rk4(S, dt, **kw)
        vel   = np.hypot(S[2], S[3])
        cerca = np.min(np.stack([np.hypot(S[0]-xm, S[1]-ym) for xm, ym in IMANES]), axis=0)
        listo = (vel < 1e-3) & (cerca < 0.4)
        if listo.any():
            final[:, quedan[listo]] = S[:, listo]
            S = S[:, ~listo]
            quedan = quedan[~listo]
            if S.shape[1] == 0:
                break
    if S.shape[1]:
        final[:, quedan] = S
    return final

def iman_final(S):
    "Devuelve 0, 1 o 2 según a qué imán quedó más cerca cada péndulo."
    d2 = np.stack([(S[0]-xm)**2 + (S[1]-ym)**2 for xm, ym in IMANES])
    return np.argmin(d2, axis=0)

### 6.1 Dos sueltas casi idénticas

Soltamos el péndulo desde un punto, y después desde otro punto separado del
primero por una distancia $\delta$ minúscula.

In [ ]:
@interact(x0=FloatSlider(min=-1.8, max=1.8, step=0.05, value=-0.75,
                         description="x₀", continuous_update=False),
          y0=FloatSlider(min=-1.8, max=1.8, step=0.05, value=-1.50,
                         description="y₀", continuous_update=False),
          delta=SelectionSlider(options=[("1e-1",1e-1), ("1e-2",1e-2), ("1e-3",1e-3),
                                         ("1e-4",1e-4), ("1e-5",1e-5), ("1e-6",1e-6)],
                                value=1e-3, description="δ"))
def dos_sueltas(x0, y0, delta):
    S0 = np.array([[x0, x0+delta], [y0, y0], [0.0, 0.0], [0.0, 0.0]])
    Sf, traj = integrar(S0, dt=0.02, tmax=500, guardar=True)
    w = iman_final(Sf)

    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    ax.plot(traj[:,0,0], traj[:,1,0], lw=0.9, color="black",  alpha=0.75,
            label=f"suelta A  → imán {w[0]+1}")
    ax.plot(traj[:,0,1], traj[:,1,1], lw=0.9, color="orange", alpha=0.9,
            label=f"suelta B  → imán {w[1]+1}")
    for i, (xm, ym) in enumerate(IMANES):
        ax.plot(xm, ym, "o", ms=17, color=COLORES[i], mec="k", mew=1.2)
        ax.annotate(str(i+1), (xm, ym), ha="center", va="center", fontsize=9)
    ax.plot(x0, y0, "k*", ms=14, zorder=6)
    ax.set_aspect("equal"); ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    igual = "MISMO imán" if w[0] == w[1] else "¡IMANES DISTINTOS!"
    ax.set_title(f"δ = {delta:.0e}   →   {igual}")
    ax.legend(loc="upper right", fontsize=8)
    plt.show()

✏️ **Para probar:** con el punto que viene por defecto, bajen $\delta$ desde
$10^{-1}$ hasta $10^{-6}$. Dos sueltas que difieren en **una millonésima**
terminan en imanes distintos.

Ningún experimentalista puede controlar la condición inicial con esa precisión.
El resultado es, en la práctica, **impredecible aunque el sistema sea
perfectamente determinista**. Ésta es la frase para dejar escrita en el pizarrón:

> **Determinista no es lo mismo que predecible.**

Prueben también con $(x_0, y_0)$ cerca de un imán, por ejemplo $(0, 1)$: ahí sí
el resultado es estable y $\delta$ no cambia nada. **No todo el plano es sensible.**
Cuál parte lo es, es justamente la próxima pregunta.

### 6.2 El mapa de cuencas de atracción

Ahora hacemos lo mismo para **todas** las condiciones iniciales de una grilla y
pintamos cada punto del color del imán donde termina.

⏱️ Con resolución 120 tarda unos 15–25 segundos. Ejecuten la celda y esperen.

In [ ]:
def calcular_cuencas(N=120, ancho=2.5, cx=0.0, cy=0.0, dt=0.05, tmax=500):
    xs = np.linspace(cx - ancho, cx + ancho, N)
    ys = np.linspace(cy - ancho, cy + ancho, N)
    X, Y = np.meshgrid(xs, ys)
    S = np.stack([X.ravel(), Y.ravel(), np.zeros(N*N), np.zeros(N*N)])
    Sf = integrar_rapido(S, dt=dt, tmax=tmax)
    return xs, ys, iman_final(Sf).reshape(N, N)

def dibujar_cuencas(xs, ys, M, titulo=""):
    from matplotlib.colors import ListedColormap
    plt.figure(figsize=(6.4, 6.4))
    plt.imshow(M, origin="lower", extent=[xs[0], xs[-1], ys[0], ys[-1]],
               cmap=ListedColormap(COLORES), vmin=-0.5, vmax=2.5,
               interpolation="nearest")
    for i, (xm, ym) in enumerate(IMANES):
        if xs[0] < xm < xs[-1] and ys[0] < ym < ys[-1]:
            plt.plot(xm, ym, "o", ms=11, color="white", mec="k", mew=1.6)
    plt.xlabel("x₀"); plt.ylabel("y₀"); plt.title(titulo); plt.grid(False)
    plt.show()

In [ ]:
# ---- vista general ----
xs, ys, M = calcular_cuencas(N=120, ancho=2.5)
dibujar_cuencas(xs, ys, M, "Cuencas de atracción — vista general (120×120)")

Cerca de cada imán la cuenca es un bloque compacto: soltás ahí, terminás ahí, sin
sorpresas. Pero entre medio los tres colores aparecen **entremezclados**.

Hagamos zoom para ver hasta dónde llega esa mezcla.

In [ ]:
# ---- zoom: cambiar 'ancho' para acercarse más ----
ancho_zoom = 0.35      # ←←← probar 1.0, 0.35, 0.1, 0.03
centro_x   = -0.75
centro_y   = -1.50

xs, ys, M = calcular_cuencas(N=120, ancho=ancho_zoom, cx=centro_x, cy=centro_y)
dibujar_cuencas(xs, ys, M, f"Zoom  (semiancho = {ancho_zoom})")

✏️ **Para probar:** bajen `ancho_zoom` de 1.0 a 0.35, a 0.1, a 0.03, ejecutando la
celda cada vez. La estructura **no se simplifica nunca**: por más que amplifiquen,
los tres colores siguen entremezclados. Eso es un **borde fractal**.

Comparen con lo que pasaría si hicieran zoom sobre el borde de un círculo: a la
tercera ampliación se vería una recta. Acá no.

> **La clave física:** la sensibilidad a las condiciones iniciales es
> **geométrica**. Vive en la estructura de las cuencas, no en la complejidad de la
> ecuación. La ecuación del péndulo magnético entra en tres renglones.

### 6.3 🧩 Ejercicio 2 — el experimento que más incomoda

Vamos a calcular el **mismo** mapa dos veces, cambiando sólo el paso de
integración `dt`. Es decir: la misma física, la misma condición inicial, sólo un
poquito más de precisión numérica en un caso.

Antes de ejecutar, **anoten su predicción**: ¿van a salir dos figuras idénticas?

In [ ]:
xs, ys, M1 = calcular_cuencas(N=90, ancho=2.5, dt=0.05)
xs, ys, M2 = calcular_cuencas(N=90, ancho=2.5, dt=0.02)

iguales = 100*(M1 == M2).mean()
print(f"Los dos mapas coinciden en el {iguales:.1f} % de los puntos.")

fig, axes = plt.subplots(1, 2, figsize=(11, 5.4))
from matplotlib.colors import ListedColormap
for ax, M, tit in zip(axes, [M1, M2], ["dt = 0.05", "dt = 0.02"]):
    ax.imshow(M, origin="lower", extent=[xs[0], xs[-1], ys[0], ys[-1]],
              cmap=ListedColormap(COLORES), vmin=-0.5, vmax=2.5,
              interpolation="nearest")
    ax.set_title(tit); ax.set_aspect("equal"); ax.grid(False)
plt.show()

for i in range(3):
    print(f"  imán {i+1}:  {100*(M1==i).mean():5.1f} %  vs {100*(M2==i).mean():5.1f} %")

**Lo que pasa:** los dos mapas diferen en torno a la mitad de los puntos —
pero las **proporciones globales** de cada color son casi idénticas (alrededor de
33 % cada una), y las manchas grandes están en el mismo lugar.

O sea:

| | ¿Reproducible? |
|---|---|
| El destino de **un punto** de la frontera | ❌ no |
| La **estructura estadística** del dibujo | ✅ sí |

Esto **no es un error numérico que haya que arreglar**. Es la misma sensibilidad
que vimos con $\delta$, pero ahora la perturbación diminuta no viene de mover la
condición inicial: viene del error de truncamiento del integrador. En la frontera
fractal, cualquier perturbación (mover el punto, cambiar `dt`, cambiar de
computadora) cambia el resultado.

> **Para discutir con los estudiantes:** ¿qué significa "hacer una predicción"
> sobre un sistema así? La respuesta es que se predicen **distribuciones**, no
> trayectorias. Es exactamente lo que hace un pronóstico del tiempo cuando dice
> "70 % de probabilidad de lluvia".

---
## 7. Para llevarse

| Sistema | ¿Solución cerrada? | ¿Predecible? |
|---|---|---|
| Péndulo linealizado | sí, elemental | totalmente |
| Péndulo exacto | sí, con funciones elípticas | totalmente |
| Péndulo forzado y disipativo | no | no siempre |
| Péndulo magnético | no | **no, en la práctica** |

Tres ideas para cerrar el día:

1. **La no linealidad, sola, no produce caos.** El péndulo exacto es no lineal y
   es perfectamente regular y predecible.
2. **Hace falta algo más:** disipación combinada con inyección de energía, o bien
   varios atractores compitiendo.
3. **Determinista ≠ predecible.** La ecuación no tiene ninguna aleatoriedad. La
   impredecibilidad aparece de la combinación entre la geometría de las cuencas y
   nuestra precisión finita al fijar la condición inicial.

**Mañana:** el mapa logístico, la ruta al caos por duplicación de período y la
constante de Feigenbaum — donde vamos a ver que estos números son *universales*,
los mismos para sistemas que no tienen nada que ver entre sí.